# Backbone Experiments
Run multiple backbone experiments with identical training/inference pipeline.


In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from dotenv import load_dotenv

project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import src.data as data
import src.inference as inference
import src.feature_cache as feature_cache
from src.models import ArcFaceHeadModel, build_backbone
from src.utils import get_device, set_seed
from src.training import fit_embeddings
from src.wandb_utils import init_wandb

# Load environment variables from .env file
load_dotenv(project_root / ".env")

RANDOM_SEED = 42
set_seed(RANDOM_SEED)

device = get_device()
device


## Base Config


In [ ]:
config = {
    # Paths
    "data_dir": Path("../data"),
    "checkpoint_dir": Path("checkpoints"),

    # Model
    #"backbone_name": "hf-hub:BVRA/MegaDescriptor-L-384",
    #"input_size": 384,
    "embedding_dim": 256,
    "hidden_dim": 512,
    "freeze_backbone": True,

    # ArcFace
    "arcface_margin": 0.5,
    "arcface_scale": 64.0,
    "dropout": 0.3,

    # Training
    "batch_size": 32,
    "learning_rate": 1e-4,
    "weight_decay": 1e-4,
    "num_epochs": 50,
    "patience": 10,
    "val_split": 0.2,
    "num_workers": 2,

    # Reproducibility
    "seed": RANDOM_SEED,

    # Normalization
    "mean": data.DEFAULT_MEAN,
    "std": data.DEFAULT_STD,
}

## Load Data


In [ ]:
train_df = data.load_train_df(config["data_dir"])
train_df, label_encoder = data.encode_labels(train_df)
num_classes = len(label_encoder.classes_)

train_data, val_data = data.train_val_split(
    train_df,
    val_split=config["val_split"],
    seed=config["seed"],
    stratify_col="ground_truth",
)

backbone_train_loader, backbone_val_loader = data.create_backbone_dataloaders(
    train_data,
    val_data,
    img_dir=config["data_dir"] / "train" / "train",
    input_size=config["input_size"],
    batch_size=config["batch_size"],
    num_workers=config["num_workers"],
    mean=config["mean"],
    std=config["std"],
)

pairs_df = data.load_test_pairs_df(config["data_dir"])
unique_images = set(pairs_df["query_image"].unique()) | set(pairs_df["gallery_image"].unique())
unique_images = sorted(list(unique_images))

test_df = pd.DataFrame({"filename": unique_images})

test_loader = data.create_test_loader(
    test_df,
    img_dir=config["data_dir"] / "test" / "test",
    input_size=config["input_size"],
    batch_size=config["batch_size"],
    num_workers=config["num_workers"],
    mean=config["mean"],
    std=config["std"],
)

print(f"Train batches: {len(backbone_train_loader)} | Val batches: {len(backbone_val_loader)} | Test batches: {len(test_loader)}")


## Submission Helper


In [ ]:
def create_submission(
    backbone,
    head_model,
    device,
    pairs_df,
    test_loader,
    output_path,
):
    names, embeddings = inference.extract_embeddings_with_names_backbone(
        backbone,
        head_model,
        test_loader,
        device,
    )
    embedding_lookup = inference.build_embedding_lookup(names, embeddings)

    similarities = inference.compute_similarity_for_pairs(pairs_df, embedding_lookup)

    submission_df = pd.DataFrame({
        "row_id": pairs_df["row_id"],
        "similarity": similarities,
    })

    submission_df.to_csv(output_path, index=False)
    return submission_df


## Run Experiments


In [ ]:
backbones = {
    "MegaDescriptor": {"model": "hf-hub:BVRA/MegaDescriptor-L-384", "input_size": 384},
    "Eva-02": {"model": "eva02_large_patch14_448.mim_m38m_ft_in22k_in1k", "input_size": 448}
    #"hf-hub:microsoft/resnet-18",
}

In [ ]:
def run_experiment(backbone_name, backbone_info,run_idx):
    print("=" * 80)
    print(f"Run {run_idx}/{len(backbones)}: {backbone_name}")

    config["backbone_name"] = backbone_info["model"]
    config["input_size"] = backbone_info["input_size"]

    backbone = build_backbone(backbone_info["model"], pretrained=True).to(device)
    backbone.eval()
    backbone_out_dim = getattr(backbone, "num_features", None)
    if backbone_out_dim is None:
        raise ValueError("Backbone output dimension not found; pass backbone_out_dim")

    cache_dir = config["checkpoint_dir"] / "embedding_cache"
    cache_dir.mkdir(exist_ok=True)
    cache_key = backbone_name.replace(":", "_").replace("/", "_")

    train_cache = cache_dir / f"train_{cache_key}.npz"
    val_cache = cache_dir / f"val_{cache_key}.npz"

    train_embeddings, train_labels = feature_cache.get_or_create_embeddings(
        train_cache,
        backbone,
        backbone_train_loader,
        device,
    )
    val_embeddings, val_labels = feature_cache.get_or_create_embeddings(
        val_cache,
        backbone,
        backbone_val_loader,
        device,
    )

    train_loader, val_loader = data.create_embedding_dataloaders(
        train_embeddings,
        train_labels,
        val_embeddings,
        val_labels,
        batch_size=config["batch_size"],
    )

    head_model = ArcFaceHeadModel(
        input_dim=backbone_out_dim,
        num_classes=num_classes,
        embedding_dim=config["embedding_dim"],
        hidden_dim=config["hidden_dim"],
        margin=config["arcface_margin"],
        scale=config["arcface_scale"],
        dropout=config["dropout"],
    ).to(device)

    param_count = sum(p.numel() for p in head_model.parameters())

    run_name = f"{backbone_name.split('/')[-1]}-arcface-{run_idx}"
    wandb = init_wandb(config, run_name=run_name, param_count=param_count)

    criterion = torch.nn.CrossEntropyLoss()

    optimizer = torch.optim.AdamW(
        [p for p in head_model.parameters() if p.requires_grad],
        lr=config["learning_rate"],
        weight_decay=config["weight_decay"],
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=5,
    )

    checkpoint_name = f"arcface_head_best_{run_idx}.pth"
    results = fit_embeddings(
        model=head_model,
        train_loader=train_loader,
        val_loader=val_loader,
        config=config,
        device=device,
        criterion=criterion,
        optimizer=optimizer,
        scheduler=scheduler,
        label_encoder=label_encoder,
        wandb_run=wandb,
        checkpoint_name=checkpoint_name,
    )

    wandb.run.summary["best_val_mAP"] = results["best_map"]
    wandb.run.summary["best_val_loss"] = results["best_val_loss"]
    wandb.run.summary["best_epoch"] = results["best_epoch"]
    wandb.run.summary["total_epochs"] = results["epochs_ran"]

    checkpoint_path = config["checkpoint_dir"] / checkpoint_name
    submission_path = config["checkpoint_dir"] / f"submission_{run_idx}.csv"
    submission_df = create_submission(
        backbone,
        head_model,
        device,
        pairs_df,
        test_loader,
        submission_path,
    )

    model_artifact = wandb.Artifact(
        name=f"arcface-head-{run_idx}",
        type="model",
        description="ArcFace head checkpoint",
    )
    model_artifact.add_file(str(checkpoint_path))
    wandb.log_artifact(model_artifact)

    submission_artifact = wandb.Artifact(
        name=f"submission-{run_idx}",
        type="submission",
        description="Competition submission file",
    )
    submission_artifact.add_file(str(submission_path))
    wandb.log_artifact(submission_artifact)

    wandb.finish()
    print(f"Finished run {run_idx}")


for run_idx, (backbone_name, backbone_info) in enumerate(backbones.items(), start=1):
    run_experiment(backbone_name, backbone_info run_idx)


In [ ]:
def run_experiment(backbone_name, backbone, run_idx):
    print("=" * 80)
    print(f"Run {run_idx}/{len(backbones)}: {backbone_name}")

    # Update config for this run
    config["backbone_name"] = backbone["model"]
    config["input_size"] = backbone["input_size"]

    train_loader, val_loader, test_loader, pairs_df, num_classes, label_encoder = load_data()

    # Init model
    model = ArcFaceModel(
        backbone_name=config["backbone_name"],
        num_classes=num_classes,
        embedding_dim=config["embedding_dim"],
        hidden_dim=config["hidden_dim"],
        margin=config["arcface_margin"],
        scale=config["arcface_scale"],
        dropout=config["dropout"],
        pretrained=True,
        freeze_backbone=config["freeze_backbone"],
    ).to(device)

    param_count = sum(p.numel() for p in model.parameters())

    # W&B init
    run_name = f"backbone-{backbone_name}"
    wandb = init_wandb(config, run_name=run_name, param_count=param_count)

    # Training components
    criterion = torch.nn.CrossEntropyLoss()

    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=config["learning_rate"],
        weight_decay=config["weight_decay"],
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=5,
    )

    # Train
    checkpoint_name = f"{run_name}.pth"
    results = fit(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        config=config,
        device=device,
        criterion=criterion,
        optimizer=optimizer,
        scheduler=scheduler,
        label_encoder=label_encoder,
        wandb_run=wandb,
        checkpoint_name=checkpoint_name,
    )

    wandb.run.summary["best_val_mAP"] = results["best_map"]
    wandb.run.summary["best_val_loss"] = results["best_val_loss"]
    wandb.run.summary["best_epoch"] = results["best_epoch"]
    wandb.run.summary["total_epochs"] = results["epochs_ran"]

    # Submission
    checkpoint_path = config["checkpoint_dir"] / checkpoint_name
    submission_path = config["checkpoint_dir"] / f"submission_{run_idx}.csv"
    submission_df = create_submission(
        model,
        device,
        pairs_df,
        test_loader,
        checkpoint_path,
        submission_path,
    )

    # W&B artifacts
    model_artifact = wandb.Artifact(
        name=f"{run_name}",
        type="model",
        description=f"model checkpoint {run_name}",
    )
    model_artifact.add_file(str(checkpoint_path))
    wandb.log_artifact(model_artifact)

    submission_artifact = wandb.Artifact(
        name=f"submission-{run_name}",
        type="submission",
        description="Competition Submission File",
    )
    submission_artifact.add_file(str(submission_path))
    wandb.log_artifact(submission_artifact)

    wandb.finish()
    print(f"Finished run {run_idx}")


for run_idx, (backbone_name, backbone) in enumerate(backbones.items(), start=1):
    run_experiment(backbone_name, backbone, run_idx)
